# logit-thermo A2-7b：本滴定のデータ集め（Pythia-1B、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは約 1.5〜2 時間。終わったら自動で切れます。
- ゲート：`docs/A2_7B_GATE.md`。Pythia-1B で 8 本の腕（なし・なし（別の種）・写しの禁止 q ＝ 1/3・2/3・1・偽薬 r ＝ 2.5%・5%・7.5%）を、温度 8 つ（0.4〜1.0）で。データ集めはモデルごとに 3 本あり、同時に回して大丈夫です。
- 計算のコード：`scripts/a2_2_collect.py`・`scripts/a2_4_collect.py`・`scripts/a2_7_collect.py` は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_7b_collect.py`（`b03ff9e2b16e2297…`）は埋め込み。
- 保存先 `A2_7B/`。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。
- **通知**：🚀・腕ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-7b 本滴定 データ集め（Pythia-1B、写し q・偽薬 r の 8 本、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-7b 本滴定 データ集め（Pythia-1B、写し q・偽薬 r の 8 本、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-7b titration pythia', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550'})
SRC22, SRC24, SRC7C = _s['SRC22'], _s['SRC24'], _s['SRC7C']
SRC7BC = '"""A2-7b（ゲート docs/A2_7B_GATE.md）のデータ集め：本滴定（写しの禁止の小数の濃さと、偽薬の濃さ 3 段）。GPU（L4）。\nscripts/a2_2_collect.py・scripts/a2_4_collect.py・scripts/a2_7_collect.py を実行したのと同じ名前空間で実行する\n（step_stats・split_doc・load_model・valid_mask・h2_of・CONT・SAMPLES・CopyIndexN・generate_ngram を変えずに使う）。\n"""\nimport gc, math, os, time\nimport numpy as np\nimport torch\n\nB7_TEMPS = [0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 1.0]\nSEED7B = 20261110\nB7_ARMS = [(\'none\', \'none\', 0.0), (\'none_b\', \'none\', 0.0),\n           (\'cq33\', \'copy\', 1 / 3), (\'cq67\', \'copy\', 2 / 3), (\'cq100\', \'copy\', 1.0),\n           (\'pl025\', \'placebo\', 0.025), (\'pl050\', \'placebo\', 0.05), (\'pl075\', \'placebo\', 0.075)]\nB7_GROUPS = {\'pythia\': (\'pythia-1b\', \'EleutherAI/pythia-1b\'), \'olmo\': (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'),\n             \'qwen\': (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\')}\nB7_N = 3\n\n\ndef gen_seed(k, s, arm):\n    return SEED7B + 1000 * k + s + (500000 if arm == \'none_b\' else 0)\n\n\ndef coin_seed(k, s):\n    return SEED7B + 900000 + 1000 * k + s\n\n\n@torch.no_grad()\ndef generate_7b(model, prompts, T, valid, gen, H2, kind, dose, rng):\n    """kind：\'none\'・\'copy\'（3 トークンの写しの候補を、各歩・各記事で確率 dose で全部禁止）・\'placebo\'（1 位が写しの候補でない歩で、\n    全部の歩 × 記事の割合 dose だけ 1 位を禁止。A2-4 の偽薬と同じ「残りの借り」）。手順は A2-7 の generate_ngram と同じ並び。"""\n    device = valid.device\n    B = len(prompts)\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    idx = [CopyIndexN(p, B7_N) for p in prompts]\n    owed = 0.0\n    rec = {k: [] for k in [\'ell\', \'cand\', \'banned\', \'banned_top1\', \'ban_mass\', \'toks\']}\n    for t in range(CONT):\n        ell = step_stats(logits, T, valid, H2)[0]                         # つまみ前の logit（本命）\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        lp = torch.log_softmax(z / T, dim=-1)\n        argmax = z.argmax(-1)\n        cands = [idx[b].candidates() for b in range(B)]\n        cand = torch.tensor([len(c) > 0 for c in cands], device=device)\n        rows, cols = [], []\n        if kind == \'copy\':\n            coin = rng.random(B) < dose                                   # くじは毎歩 B 枚（候補がなくても引く）\n            for b, c in enumerate(cands):\n                if c and coin[b]:\n                    rows += [b] * len(c); cols += list(c)\n        elif kind == \'placebo\':\n            am = argmax.cpu().numpy()\n            elig = np.array([int(am[b]) not in set(cands[b]) for b in range(B)])\n            owed += dose * B\n            q = min(1.0, owed / elig.sum()) if elig.any() else 0.0\n            pick = (rng.random(B) < q) & elig\n            owed -= int(pick.sum())\n            for b in np.nonzero(pick)[0]:\n                rows.append(int(b)); cols.append(int(am[b]))\n        banned = torch.zeros(B, dtype=torch.bool, device=device)\n        banned_top1 = torch.zeros(B, dtype=torch.bool, device=device)\n        ban_mass = torch.zeros(B, dtype=torch.float64, device=device)\n        zp = z.clone()\n        if rows:\n            r = torch.tensor(rows, device=device); c = torch.tensor(cols, device=device)\n            zp[r, c] = float(\'-inf\')\n            banned[r] = True\n            banned_top1[r[c == argmax[r]]] = True\n            ban_mass.index_add_(0, r, torch.exp(lp[r, c]))\n        p = torch.softmax(zp / T, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        for b in range(B):\n            idx[b].push(int(nxt[b, 0]))\n        for k, v in [(\'ell\', ell), (\'cand\', cand), (\'banned\', banned), (\'banned_top1\', banned_top1), (\'ban_mass\', ban_mass),\n                     (\'toks\', nxt[:, 0])]:\n            rec[k].append(v)\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    res = {}\n    for k, v in rec.items():\n        a = torch.stack(v, 1)\n        res[k] = a.cpu().numpy().astype(np.int32 if k == \'toks\' else (np.bool_ if a.dtype == torch.bool else np.float32))\n    return res\n\n\n@torch.no_grad()\ndef teacher_7b(model, prompts, humans, valid, H2, device):\n    """→ (nD, nT, L)：人の続きでの ℓ（順位 H 以降）を B7_TEMPS で。"""\n    out = np.zeros((len(prompts), len(B7_TEMPS), CONT), np.float32)\n    for j, (pr, hu) in enumerate(zip(prompts, humans)):\n        x = torch.tensor([list(pr) + list(hu)], device=device)\n        lg = model(x).logits[0, len(pr) - 1: len(pr) - 1 + CONT].float()\n        for k, T in enumerate(B7_TEMPS):\n            out[j, k] = step_stats(lg, T, valid, H2)[0].float().cpu().numpy()\n    return out\n\n\ndef run_7b(name, hf_id, docs, outdir, device, notify=print):\n    todo = [a for a in B7_ARMS if not os.path.exists(f\'{outdir}/{name}@{a[0]}.npz\')]\n    if not todo and os.path.exists(f\'{outdir}/{name}@tf.npz\'):\n        print(name, \'はもう全部終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nD = len(use)\n    plen = {len(pairs[i][0]) for i in use}\n    assert len(plen) == 1, plen\n    prompts = [pairs[i][0] for i in use]\n    humans = [pairs[i][1] for i in use]\n    if not os.path.exists(f\'{outdir}/{name}@tf.npz\'):\n        tf = teacher_7b(model, prompts, humans, valid, H2, device)\n        np.savez_compressed(f\'{outdir}/{name}@tf.npz.part.npz\', tf=tf, temps=np.array(B7_TEMPS), use=np.array(use))\n        os.replace(f\'{outdir}/{name}@tf.npz.part.npz\', f\'{outdir}/{name}@tf.npz\')\n        print(f\'  {name} 人の続きの ℓ おわり（{time.time() - t0:.0f} 秒）\')\n    keys_f, keys_b = [\'ell\', \'ban_mass\'], [\'cand\', \'banned\', \'banned_top1\']\n    for arm, kind, dose in todo:\n        ta = time.time()\n        G = {k: np.zeros((len(B7_TEMPS), SAMPLES, nD, CONT), np.float32) for k in keys_f}\n        G.update({k: np.zeros((len(B7_TEMPS), SAMPLES, nD, CONT), np.bool_) for k in keys_b})\n        toks = np.zeros((len(B7_TEMPS), SAMPLES, nD, CONT), np.int32)\n        for k, T in enumerate(B7_TEMPS):\n            for s in range(SAMPLES):\n                g = torch.Generator(device=device).manual_seed(gen_seed(k, s, arm))\n                rng = np.random.default_rng(coin_seed(k, s))\n                r = generate_7b(model, prompts, T, valid, g, H2, kind, dose, rng)\n                for kk in keys_f + keys_b:\n                    G[kk][k, s] = r[kk]\n                toks[k, s] = r[\'toks\']\n        path = f\'{outdir}/{name}@{arm}.npz\'\n        np.savez_compressed(path + \'.part.npz\', toks=toks, temps=np.array(B7_TEMPS), use=np.array(use), eos=np.array(eos),\n                            prompts=np.array(prompts, np.int32), arm=arm, kind=kind, dose=dose, seconds=time.time() - ta,\n                            **{f\'g_{k}\': v for k, v in G.items()})\n        os.replace(path + \'.part.npz\', path)\n        notify(f\'{name}@{arm} おわり（{time.time() - ta:.0f} 秒、通算 {time.time() - t0:.0f} 秒）\')\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef self_test_7b(device):\n    """小さなランダムのモデルで：copy q ＝ 1 が A2-7 の n ＝ 3 と同じトークン・ℓ。copy q ＝ 0 と placebo r ＝ 0 が「なし」と同じ。\n    placebo の禁止の割合が r に近い。copy q ＝ 1/3 で、候補のある歩のうち禁止した割合が 1/3 に近い。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    global CONT\n    keep = CONT\n    CONT = 60\n    try:\n        prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4, [20, 21, 22, 23] * 4]\n        for T in [0.4, 1.0]:\n            g1 = torch.Generator(device=device).manual_seed(3)\n            a = generate_ngram(model, prompts, T, valid, g1, 33, 3)\n            g2 = torch.Generator(device=device).manual_seed(3)\n            b = generate_7b(model, prompts, T, valid, g2, 33, \'copy\', 1.0, np.random.default_rng(0))\n            assert (a[\'toks\'] == b[\'toks\']).all() and np.allclose(a[\'ell\'], b[\'ell\']), (\'q ＝ 1 が n ＝ 3 と違う\', T)\n            outs = []\n            for kind in (\'none\', \'copy\', \'placebo\'):\n                g = torch.Generator(device=device).manual_seed(4)\n                outs.append(generate_7b(model, prompts, T, valid, g, 33, kind, 0.0, np.random.default_rng(1))[\'toks\'])\n            assert (outs[0] == outs[1]).all() and (outs[0] == outs[2]).all(), (\'濃さ 0 が「なし」と違う\', T)\n        CONT = 300\n        g = torch.Generator(device=device).manual_seed(5)\n        r = generate_7b(model, prompts * 4, 1.0, valid, g, 33, \'placebo\', 0.075, np.random.default_rng(2))\n        rate = r[\'banned\'].mean()\n        assert abs(rate - 0.075) < 0.01 and (r[\'banned\'] == r[\'banned_top1\']).all(), rate\n        g = torch.Generator(device=device).manual_seed(6)\n        r = generate_7b(model, prompts * 16, 0.1, valid, g, 33, \'copy\', 1 / 3, np.random.default_rng(3))   # ほぼ貪欲・64 本にして写しの候補を増やす\n        nc = int(r[\'cand\'].sum())\n        fr = r[\'banned\'][r[\'cand\']].mean()\n        # 2026-10-29 22:50 の Colab で、16 本 × 300 歩では候補が 54 歩しかなく止まった（禁止の割合 0.407 は 1/3 から標準誤差 1.2 個ぶん）。\n        # 本数を増やし、許す幅を標準誤差の 4 倍にした\n        assert nc >= 50 and abs(fr - 1 / 3) < 4 * math.sqrt((1 / 3) * (2 / 3) / nc), (fr, nc)\n        assert not (r[\'banned\'] & ~r[\'cand\']).any()\n        assert all(T in B7_TEMPS for T in [0.8, 0.85, 0.9]) and len({gen_seed(k, s, a) for k in range(8) for s in range(2) for a in (\'none\', \'none_b\')}) == 32\n    finally:\n        CONT = keep\n    return \'OK\'\n'
assert hashlib.sha256(SRC7BC.encode()).hexdigest() == 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 8 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
GROUP = 'pythia'
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_7B', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    name, hf_id = B7_GROUPS[GROUP]
    run_7b(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_7B', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(gate='docs/A2_7B_GATE.md', group=GROUP, a2_2_collect_sha256='e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', a2_4_collect_sha256='6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57',
                   a2_7_collect_sha256='8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', a2_7b_collect_sha256='b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', torch=torch.__version__,
                   transformers=transformers.__version__, python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0),
                   matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_7B' + f'/run_meta_{GROUP}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-7b のデータ集め（Pythia-1B）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-7b の Pythia-1B を A2_7B に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
